# Chunk Documents

This notebook corresponds to `src/ingestion/chunk_documents.py`.

Purpose:

```text
Data/documents.json  →  Data/chunks.json
```

It splits each document into overlapping word-based chunks. These chunks are later embedded and stored in the vector database.

In [ ]:
from pathlib import Path
import json
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DOCUMENTS_PATH = PROJECT_ROOT / "Data" / "documents.json"
CHUNKS_PATH = PROJECT_ROOT / "Data" / "chunks.json"

print("Documents path exists:", DOCUMENTS_PATH.exists())
print("Chunks output path:", CHUNKS_PATH)

In [ ]:
with DOCUMENTS_PATH.open("r", encoding="utf-8") as f:
    documents = json.load(f)

print("Number of documents:", len(documents))
print("Document keys:", documents[0].keys() if documents else None)

In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    words = text.split()

    if chunk_size <= overlap:
        raise ValueError("chunk_size must be larger than overlap.")

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunk = " ".join(chunk_words)

        if len(chunk.strip()) > 100:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
def build_chunks(
    documents_path: Path,
    output_path: Path,
    chunk_size: int = 500,
    overlap: int = 100,
) -> list:
    with documents_path.open("r", encoding="utf-8") as f:
        documents = json.load(f)

    all_chunks = []

    for doc in documents:
        chunks = chunk_text(
            doc["content"],
            chunk_size=chunk_size,
            overlap=overlap,
        )

        for i, chunk in enumerate(chunks, start=1):
            chunk_id = f"{doc['doc_id']}_chunk_{i:03d}"

            all_chunks.append({
                "chunk_id": chunk_id,
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "source_html": doc["source_html"],
                "text": chunk,
            })

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)

    return all_chunks

In [ ]:
chunks = build_chunks(
    documents_path=DOCUMENTS_PATH,
    output_path=CHUNKS_PATH,
    chunk_size=500,
    overlap=100,
)

print(f"Saved {len(chunks)} chunks to {CHUNKS_PATH}")

In [ ]:
chunks_df = pd.DataFrame([
    {
        "chunk_id": c["chunk_id"],
        "doc_id": c["doc_id"],
        "title": c["title"],
        "text_length_chars": len(c["text"]),
        "text_length_words": len(c["text"].split()),
    }
    for c in chunks
])

chunks_df.head(20)

In [ ]:
chunks_df[["text_length_chars", "text_length_words"]].describe()

In [ ]:
if chunks:
    print("Sample chunk ID:", chunks[0]["chunk_id"])
    print("Doc ID:", chunks[0]["doc_id"])
    print("Text preview:")
    print(chunks[0]["text"][:1500])